## 第2步 奖励模型（RewardModel）微调
8 x 16Gib P100 GPU 

### 定义通用的参数

In [1]:
import argparse

args = argparse.ArgumentParser()

args.local_rank = 0
args.data_path = ["/mnt/Dahoas/rm-static"] 
args.data_split = "2,4,4" 
args.data_output_path = '/tmp/data_files'
args.max_seq_len = 256
args.num_train_epochs = 1 
args.seed = 1234 
args.per_device_train_batch_size = 1
args.per_device_eval_batch_size = 1
args.print_loss = True

## 生成模型

In [2]:
from transformers import LlamaModel, LlamaTokenizer

from torch.utils.data import DataLoader, RandomSampler, SequentialSampler

args.model_name_or_path = '/mnt/chinese-llama-2-1.3b' #'/mnt/chinese-llama-2-7b'

tokenizer = LlamaTokenizer.from_pretrained(args.model_name_or_path)
critic_model = LlamaModel.from_pretrained(args.model_name_or_path, device_map="auto", offload_folder = '/tmp').eval()

[2023-12-22 16:39:21,302] [INFO] [real_accelerator.py:161:get_accelerator] Setting ds_accelerator to cuda (auto detect)


## 组织训练数据

In [3]:
from dschat.utils.data.data_utils import create_prompt_dataset, DataCollatorReward

train_phase = 2
train_dataset, eval_dataset = create_prompt_dataset(
    args.local_rank, args.data_path, args.data_split,
    args.data_output_path, train_phase, args.seed, tokenizer,
    args.max_seq_len)

Creating prompt dataset ['/mnt/Dahoas/rm-static'], reload=False
Creating dataset Dahoas_rm_static for train_phase=2 size=30502
Creating dataset Dahoas_rm_static for train_phase=2 size=2041


## 显示一个样本的信息

In [33]:
torch.set_printoptions(linewidth=90)

In [5]:
# class PromptDataset在第2步的输出格式是：
# return self.chosen_dataset[idx]["input_ids"], self.chosen_dataset[idx]["attention_mask"], \
#                self.reject_dataset[idx]["input_ids"], self.reject_dataset[idx]["attention_mask"]

for chosen_input_ids, chosen_attention_mask, reject_input_ids, reject_attention_mask in train_dataset:
    print("选择数据样本输入(chosen_input) ------------------------------")
    inputs = tokenizer.decode(chosen_input_ids[0], skip_special_tokens=True)
    print(f'原文：{inputs}')
    print(f'input_ids：{chosen_input_ids}')
    print(f'attention_mask：{chosen_attention_mask}')

    print("拒绝数据样本输入(reject_input_ids) ------------------------------")
    print(reject_input_ids.shape)
    inputs = tokenizer.decode(reject_input_ids[0], skip_special_tokens=True)
    print(f'原文：{inputs}')
    print(f'input_ids：{reject_input_ids}')
    print(f'attention_mask：{reject_attention_mask}')
     
    break

chosen_id = chosen_input_ids[0]
rejected_id = reject_input_ids[0]

选择数据样本输入(chosen_input) ------------------------------
原文：

Human: Are miniature roses a good gift to give? I wonder if they might be hardy, or tend to die easily.

Assistant: Miniature roses might seem like an obvious choice, but they are usually bred specifically to be very small, so their bushiness is not an accurate measure of their care requirements. In general, the minimum size of a rose is 3 inches in diameter, with many roses thriving at 2 inches.<|endoftext|>
input_ids：tensor([[    1, 29871,    13,    13, 29950,  7889, 29901,  4683, 20629,  1535,
         14652,   267,   263,  1781, 19797,   304,  2367, 29973,   306,  4997,
           565,   896,  1795,   367,  2898, 29891, 29892,   470, 10331,   304,
           762,  5948, 29889,    13,    13,  7900, 22137, 29901,   341,  2172,
          1535, 14652,   267,  1795,  2833,   763,   385,  6924,  7348, 29892,
           541,   896,   526,  5491,   289,  1127, 10816,   304,   367,  1407,
          2319, 29892,   577,  1009, 27089, 

## 模仿RewardModel中的forware()

In [6]:
from torch import nn
import torch

data_collator = DataCollatorReward()
train_dataloader = DataLoader(train_dataset, collate_fn=data_collator, 
                              batch_size=args.per_device_train_batch_size)

loss = 0
chosen_mean_scores = []
rejected_mean_scores = []

v_head = nn.Linear(critic_model.config.hidden_size, 1, bias=False)
for step, batch in enumerate(train_dataloader):
    transformer_outputs = critic_model(**batch)
          
    hidden_states = transformer_outputs[0]
    rewards = v_head(hidden_states).squeeze(-1)

    chosen_reward = rewards[0]   #正样本的奖励
    rejected_reward = rewards[1]  #负样本的奖励

    PAD_ID = tokenizer.eos_token_id
    compute_fp32_loss = True

    seq_len = len(chosen_id)

    c_inds = (chosen_id == PAD_ID).nonzero()   #找出所有与padding token相等的token索引
    c_ind = c_inds[0].item() if len(c_inds) > 0 else seq_len
    
    check_divergence = (chosen_id != rejected_id).nonzero()
    if len(check_divergence) == 0: #两组样本完全相同
        end_ind = rejected_reward.size(-1)
        divergence_ind = end_ind - 1
        r_ind = c_ind
    else:
        r_inds = (rejected_id == PAD_ID).nonzero()  #找出所有与padding token相等的token索引
        r_ind = r_inds[0].item() if len(r_inds) > 0 else seq_len
        end_ind = max(c_ind, r_ind)
        divergence_ind = check_divergence[0]
        
    assert divergence_ind > 0
    c_truncated_reward = chosen_reward[divergence_ind:end_ind]
    r_truncated_reward = rejected_reward[divergence_ind:end_ind]
    
    chosen_mean_scores.append(chosen_reward[c_ind - 1]) 
    rejected_mean_scores.append(rejected_reward[r_ind - 1])

    if compute_fp32_loss:
        c_truncated_reward = c_truncated_reward.float()
        r_truncated_reward = r_truncated_reward.float()

    loss += -torch.nn.functional.logsigmoid(c_truncated_reward - r_truncated_reward).mean()

    break

In [34]:
print(f'c_inds={c_inds.T}\n{c_ind=}\ncheck_divergence={check_divergence.T}')
print(f'r_inds={r_inds.T}\n{r_ind=}')
print(f'{len(check_divergence)=}\ndivergence_ind:end_ind={divergence_ind}:{end_ind}')    
print(f'{c_truncated_reward=}\n{r_truncated_reward=}')

c_inds=tensor([[110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125,
         126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141,
         142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157,
         158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173,
         174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189,
         190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205,
         206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221,
         222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237,
         238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253,
         254, 255]])
c_ind=110
check_divergence=tensor([[ 38,  39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  50,  51,  52,  53,  54,
          55,  56,  57,  58,  59,  60,  61,  62,  63,  

In [19]:
c_truncated_reward - r_truncated_reward

tensor([ 0.5885, -0.1186, -0.6686, -1.2984, -0.6147, -0.2737, -0.0172, -0.0285,
        -0.2998, -0.2756,  0.2909,  0.4053,  0.0807,  0.3030,  0.0845, -0.2796,
         0.9808,  0.7462,  1.5627, -0.0877,  0.7562,  0.1602,  0.8484,  0.4553,
         0.7903,  0.2628,  0.4742, -0.2920,  0.3334,  1.2558,  0.5666,  0.3720,
        -0.0065,  0.3360,  0.0466,  0.5325,  0.5626, -0.0818,  0.1017,  0.7732,
         0.0323, -0.0839,  0.6885, -0.0886,  0.7526, -0.1446,  0.1867,  0.6964,
         0.1482,  0.5081,  1.0480,  1.1984,  0.9700,  1.0034,  0.6239,  0.6141,
         0.0364,  0.3501,  2.1474,  1.2215,  0.7283,  1.1831,  0.7274,  1.3638,
         1.0879,  0.4282,  0.1823, -0.0613,  0.5656,  0.1023,  0.4338,  0.2397],
       grad_fn=<SubBackward0>)

In [20]:
torch.nn.functional.logsigmoid(c_truncated_reward - r_truncated_reward)

tensor([-0.4416, -0.7542, -1.0823, -1.5398, -1.0470, -0.8393, -0.7018, -0.7075,
        -0.8542, -0.8404, -0.5583, -0.5109, -0.6536, -0.5531, -0.6518, -0.8427,
        -0.3185, -0.3881, -0.1903, -0.7380, -0.3849, -0.6162, -0.3563, -0.4912,
        -0.3741, -0.5704, -0.4839, -0.8498, -0.5403, -0.2507, -0.4494, -0.5244,
        -0.6964, -0.5392, -0.6701, -0.4619, -0.4509, -0.7349, -0.6436, -0.3795,
        -0.6771, -0.7360, -0.4070, -0.7384, -0.3861, -0.7681, -0.6042, -0.4044,
        -0.6218, -0.4710, -0.3006, -0.2637, -0.3214, -0.3124, -0.4291, -0.4325,
        -0.6751, -0.5333, -0.1105, -0.2584, -0.3939, -0.2672, -0.3942, -0.2277,
        -0.2904, -0.5018, -0.6062, -0.7243, -0.4498, -0.6433, -0.4996, -0.5805],
       grad_fn=<LogSigmoidBackward0>)

In [18]:
loss += -torch.nn.functional.logsigmoid(c_truncated_reward - r_truncated_reward).mean()
loss

tensor(0.5515, grad_fn=<NegBackward0>)

## 分别计算正负样本的积分

In [24]:
chosen_mean_scores = []
rejected_mean_scores = []

chosen_mean_scores.append(chosen_reward[c_ind - 1])  #use the end score for reference
rejected_mean_scores.append(rejected_reward[r_ind - 1])

print(f'{chosen_mean_scores=}\n{rejected_mean_scores=}')

chosen_mean_scores=[tensor(-0.2231, grad_fn=<SelectBackward0>)]
rejected_mean_scores=[tensor(-0.0367, grad_fn=<SelectBackward0>)]


## 计算奖励值 - 评论家模型用

In [8]:
hidden_states = transformer_outputs[0]
values = v_head(hidden_states).squeeze(-1)

print(f'{values.size()}')
print(values)
# values[0] chosen
# value[1] reject

torch.Size([2, 256])
tensor([[-1.9853e-01, -6.3874e-02,  2.0919e-01,  1.6003e-01,  4.9230e-02,
         -2.6846e-02, -1.4436e-01, -1.2494e-01, -2.6066e-01, -4.4779e-01,
          6.1277e-01, -2.1994e-01,  2.9735e-01, -2.2163e-01, -1.4346e-01,
          3.3157e-01, -4.3632e-01, -1.1632e-01,  1.2759e-01, -3.9894e-01,
         -4.0432e-01, -8.3180e-02, -6.1620e-02, -4.1930e-02,  1.9904e-01,
          4.1293e-01, -1.2387e-01,  1.1170e-01, -3.1630e-01,  4.3070e-02,
         -5.5780e-01, -2.5543e-01, -2.7780e-01, -7.4848e-01, -4.6634e-01,
          8.1900e-01, -2.9349e-01, -4.9402e-02, -8.5117e-01,  1.6200e-02,
         -6.9297e-02,  3.5559e-01, -6.6347e-02,  2.7630e-01,  2.1650e-02,
         -2.4748e-01, -1.0590e+00, -5.9320e-01, -1.9769e-01, -7.4118e-01,
         -3.0098e-01,  2.6743e-01,  2.0606e-01,  3.5928e-01, -1.7312e-01,
          2.1068e-02,  2.1849e-02,  7.1694e-02,  7.2260e-01,  1.1137e-01,
         -2.1492e-01, -2.0388e-01,  8.4562e-02,  1.0903e-02,  1.5940e-01,
          2.5677e